# 項目11: 個別銘柄による純化テーマファクター複製ポートフォリオ

この notebook は、項目10の事前計算pklを主入力として、個別銘柄のロングショート・ウェイトで純化テーマファクターリターンを複製する履歴型ポートフォリオを構築する。

単日スナップショット主導ではなく、`purified_snapshots.pkl` の各日付を順に処理し、最初から次の形の履歴ウェイトを作る。

```text
mimicking_weights
  index   = MultiIndex(date, security_id)
  columns = theme_id
```

各テーマ `k` について、既定では以下の制約を満たすリスク最小ポートフォリオを作る。

\[
1'w_k=0,\qquad X_t'w_k=0,\qquad z_{k,t}'w_k=1,\qquad Z_{-k,t}'w_k=0
\]

目的関数は対角リスク近似を使う。

\[
\min_{w_k}\;\frac{1}{2}w_k'\Sigma_t w_k,
\qquad
\Sigma_t=\operatorname{diag}(\widehat{\sigma}^2_{e,t}+\mathrm{ridge})
\]

`\widehat{\sigma}^2_{e,t}` は項目10の `theme_return_history.pkl` に含まれる `theme_residuals_e` から、日付 `t` 以前の履歴で推定する。複製精度は `w_{t-1}` を `t` のリターンへ適用して評価する。

## 必要データ形式

| オブジェクト | 形式 | 必須内容 | 用途 |
|---|---:|---|---|
| `purified_snapshots.pkl` | dict | `{date: snapshot}`。各 `snapshot` に `X_t`, `Z_t` | 日付別のBarraエクスポージャと純化テーマエクスポージャ |
| `theme_return_history.pkl` | dict | `theme_returns_g`, `barra_residuals_u`, `theme_residuals_e`, `barra_factor_returns_f` | 複製対象、残差リターン、リスク推定、寄与分解 |
| `precompute_metadata.pkl` | dict | 項目10の事前計算メタデータ | 設定確認用 |
| `stock_returns` | wide | index=営業日, columns=`security_id` | トータルリターンでの任意検証 |

`stock_returns` は項目10の事前計算pklには含まれないため、必要な場合は `paths["stock_returns"]` または `existing_inputs["stock_returns"]` に指定する。未指定でも Barra 残差リターンでの複製検証は実行できる。

## ユーザ設定セル

`precompute_dir` は項目10で作成した事前計算pklの保存先を指定する。履歴評価が主経路なので、既定では `RUN_PRECOMPUTED_HISTORY=True`、`RUN_ASOF_SNAPSHOT=False` とする。

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
from IPython.display import display


@dataclass(frozen=True)
class FactorMimickingConfig:
    precompute_dir: str | Path = "data/phase_b/precomputed/item10"
    asof_date: str | pd.Timestamp = "YYYY-MM-DD"
    neutralize_other_themes: bool = True
    pinv_rcond: float = 1e-10
    ridge: float = 1e-8
    constraint_tol: float = 1e-6
    min_abs_target_exposure: float = 1e-12
    risk_lookback_days: int = 60
    min_risk_observations: int = 2
    specific_variance_floor: float = 1e-10
    risk_include_asof_residual: bool = True
    continue_on_error: bool = True
    evaluation_start: str | pd.Timestamp | None = None
    evaluation_end: str | pd.Timestamp | None = None
    annualization_days: int = 252


PROJECT_ROOT = Path.cwd()
cfg = FactorMimickingConfig()

RUN_PRECOMPUTED_HISTORY = True
RUN_ASOF_SNAPSHOT = False
RUN_FACTOR_CONTRIBUTION_CHECK = True

paths = {
    "stock_returns": "data/phase_b/stock_returns.pkl",
}

# 既にNotebook上にDataFrameやdictを作っている場合はここへ渡す。
# 例: existing_inputs = {"stock_returns": stock_returns}
existing_inputs: dict[str, Any] = {}


## 基本ユーティリティ

In [ ]:
PRECOMPUTE_FILES = {
    "purified_snapshots": "purified_snapshots.pkl",
    "theme_return_history": "theme_return_history.pkl",
    "metadata": "precompute_metadata.pkl",
}

REQUIRED_SNAPSHOT_KEYS = ["X_t", "Z_t"]
REQUIRED_THEME_HISTORY_KEYS = [
    "theme_returns_g",
    "barra_residuals_u",
    "theme_residuals_e",
    "barra_factor_returns_f",
]


def _as_date(value: str | pd.Timestamp) -> pd.Timestamp:
    return pd.Timestamp(value).normalize()


def _normalise_date_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = pd.to_datetime(out.index).normalize()
    return out.sort_index()


def _normalise_datetime_key_dict(values: Mapping[Any, Any]) -> dict[pd.Timestamp, Any]:
    return {_as_date(k): v for k, v in values.items()}


def _resolve_path(path: str | Path | None) -> Path | None:
    if path is None or str(path).strip() == "":
        return None
    p = Path(path)
    return p if p.is_absolute() else PROJECT_ROOT / p


def _load_pickle(path: str | Path) -> Any:
    p = _resolve_path(path)
    if p is None:
        raise ValueError("path is empty.")
    if not p.exists():
        raise FileNotFoundError(f"Input file not found: {p}")
    return pd.read_pickle(p)


def _load_input(name: str, required: bool = True) -> Any:
    if name in existing_inputs:
        return existing_inputs[name]
    p = _resolve_path(paths.get(name))
    if p is not None and p.exists():
        return _load_pickle(p)
    if required:
        raise FileNotFoundError(
            f"{name} が見つかりません。paths['{name}'] または existing_inputs['{name}'] を指定してください。"
        )
    return None


def _empty_weight_frame() -> pd.DataFrame:
    idx = pd.MultiIndex.from_arrays([[], []], names=["date", "security_id"])
    return pd.DataFrame(index=idx)


def _safe_condition_number(matrix: np.ndarray) -> float:
    if matrix.size == 0:
        return np.nan
    try:
        return float(np.linalg.cond(matrix))
    except np.linalg.LinAlgError:
        return np.inf


def _safe_matrix_rank(matrix: np.ndarray) -> int:
    if matrix.size == 0:
        return 0
    return int(np.linalg.matrix_rank(matrix))


def _ensure_str_index_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = out.index.astype(str)
    out.columns = out.columns.astype(str)
    return out


def load_item10_precomputed_bundle(precompute_dir: str | Path) -> dict[str, Any]:
    base = _resolve_path(precompute_dir)
    if base is None:
        raise ValueError("precompute_dir is empty.")
    paths_local = {name: base / filename for name, filename in PRECOMPUTE_FILES.items()}
    missing_files = [str(p) for p in paths_local.values() if not p.exists()]
    if missing_files:
        raise FileNotFoundError("項目10の事前計算pklが不足しています: " + ", ".join(missing_files))

    bundle = {name: pd.read_pickle(path) for name, path in paths_local.items()}
    if not isinstance(bundle["purified_snapshots"], Mapping):
        raise TypeError("purified_snapshots.pkl must contain a dict-like object.")
    if not isinstance(bundle["theme_return_history"], Mapping):
        raise TypeError("theme_return_history.pkl must contain a dict-like object.")
    if not isinstance(bundle["metadata"], Mapping):
        raise TypeError("precompute_metadata.pkl must contain a dict-like object.")

    snapshots = _normalise_datetime_key_dict(bundle["purified_snapshots"])
    if not snapshots:
        raise ValueError("purified_snapshots.pkl is empty.")
    for d, snap in list(snapshots.items())[:3]:
        if not isinstance(snap, Mapping):
            raise TypeError(f"snapshot for {d.date()} must be dict-like.")
        missing = [k for k in REQUIRED_SNAPSHOT_KEYS if k not in snap]
        if missing:
            raise KeyError(f"snapshot for {d.date()} is missing keys: {missing}")

    theme_history = dict(bundle["theme_return_history"])
    missing_history = [k for k in REQUIRED_THEME_HISTORY_KEYS if k not in theme_history]
    if missing_history:
        raise KeyError(f"theme_return_history.pkl is missing keys: {missing_history}")

    for key in REQUIRED_THEME_HISTORY_KEYS:
        if not isinstance(theme_history[key], pd.DataFrame):
            raise TypeError(f"theme_return_history['{key}'] must be a DataFrame.")
        theme_history[key] = _normalise_date_index(theme_history[key])
        theme_history[key].columns = theme_history[key].columns.astype(str)

    return {
        "purified_snapshots": snapshots,
        "theme_return_history": theme_history,
        "metadata": dict(bundle["metadata"]),
        "paths": paths_local,
    }


## リスク推定

項目10の `theme_residuals_e` から日付別の銘柄固有分散を推定し、対角リスク行列を作る。`risk_include_asof_residual=True` の場合は `t` 以前、`False` の場合は `t` より前の残差だけを使う。

In [ ]:
def estimate_specific_variance_from_theme_residuals(
    theme_residuals_e: pd.DataFrame,
    asof_date: str | pd.Timestamp,
    securities: pd.Index,
    cfg: FactorMimickingConfig,
) -> tuple[pd.Series, pd.Series]:
    residuals = _normalise_date_index(theme_residuals_e)
    residuals.columns = residuals.columns.astype(str)
    d = _as_date(asof_date)
    if cfg.risk_include_asof_residual:
        window = residuals.loc[residuals.index <= d]
    else:
        window = residuals.loc[residuals.index < d]
    window = window.tail(cfg.risk_lookback_days)
    if len(window) < cfg.min_risk_observations:
        raise ValueError(
            f"specific variance estimation needs at least {cfg.min_risk_observations} observations; "
            f"got {len(window)} for {d.date()}."
        )

    securities = pd.Index(securities.astype(str))
    aligned = window.reindex(columns=securities)
    obs_count = aligned.notna().sum(axis=0)
    variance = aligned.var(axis=0, ddof=1)
    valid = obs_count >= cfg.min_risk_observations
    variance = variance.where(valid)
    variance = variance.replace([np.inf, -np.inf], np.nan).dropna()
    variance = variance.clip(lower=cfg.specific_variance_floor)
    if variance.empty:
        raise ValueError(f"No valid specific variance estimates for {d.date()}.")
    return variance.astype(float), obs_count.reindex(variance.index).astype(int)


def risk_matrix_from_specific_variance(specific_variance: pd.Series, cfg: FactorMimickingConfig) -> pd.DataFrame:
    s = pd.Series(specific_variance).astype(float)
    s.index = s.index.astype(str)
    valid = s.notna() & np.isfinite(s.to_numpy(dtype=float)) & (s > 0)
    s = s.loc[valid].clip(lower=cfg.specific_variance_floor) + cfg.ridge
    if s.empty:
        raise ValueError("specific_variance の有効な銘柄集合が空です。")
    return pd.DataFrame(np.diag(s.to_numpy(dtype=float)), index=s.index, columns=s.index)


## 複製ポートフォリオの解析解

制約行列を `C_k = [1, X_t, Z_{-k,t}, z_{k,t}]`、目標を `d_k = [0, 0, ..., 0, 1]` として、一般化逆行列でリスク最小解を求める。

In [ ]:
def _align_model_inputs(
    X_t: pd.DataFrame,
    Z_t: pd.DataFrame,
    Sigma_t: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    X = _ensure_str_index_columns(X_t).astype(float)
    Z = _ensure_str_index_columns(Z_t).astype(float)
    Sigma = Sigma_t.copy().astype(float)
    Sigma.index = Sigma.index.astype(str)
    Sigma.columns = Sigma.columns.astype(str)

    common = X.index.intersection(Z.index).intersection(Sigma.index).intersection(Sigma.columns)
    if common.empty:
        raise ValueError("X_t, Z_t, Sigma_t の共通銘柄が空です。")

    X = X.loc[common]
    Z = Z.loc[common]
    Sigma = Sigma.loc[common, common]
    finite_x = np.isfinite(X.to_numpy(dtype=float)).all(axis=1)
    finite_z = np.isfinite(Z.to_numpy(dtype=float)).all(axis=1)
    finite_sigma = np.isfinite(Sigma.to_numpy(dtype=float)).all(axis=1)
    diag_ok = np.isfinite(np.diag(Sigma.to_numpy(dtype=float))) & (np.diag(Sigma.to_numpy(dtype=float)) > 0)
    valid = pd.Index(common[finite_x & finite_z & finite_sigma & diag_ok])
    if valid.empty:
        raise ValueError("欠損除外後の有効銘柄が空です。")

    X = X.loc[valid]
    Z = Z.loc[valid]
    Sigma = Sigma.loc[valid, valid]
    Sigma = (Sigma + Sigma.T) / 2.0
    return X, Z, Sigma


def solve_one_mimicking_portfolio(
    X_t: pd.DataFrame,
    Z_t: pd.DataFrame,
    Sigma_t: pd.DataFrame,
    theme_id: str,
    cfg: FactorMimickingConfig,
) -> dict[str, Any]:
    if theme_id not in Z_t.columns.astype(str):
        raise KeyError(f"theme_id={theme_id} is not in Z_t columns.")

    X, Z, Sigma = _align_model_inputs(X_t, Z_t, Sigma_t)
    theme_id = str(theme_id)
    target_norm = float(np.linalg.norm(Z[theme_id].to_numpy(dtype=float)))
    if target_norm <= cfg.min_abs_target_exposure:
        raise ValueError(f"target theme exposure is effectively zero: {theme_id}")

    constraint_parts = [pd.Series(1.0, index=X.index, name="net").to_frame()]
    constraint_parts.append(X.add_prefix("barra:"))

    other_themes = [c for c in Z.columns.astype(str) if c != theme_id]
    if cfg.neutralize_other_themes and other_themes:
        constraint_parts.append(Z.loc[:, other_themes].add_prefix("other_theme:"))

    target_name = f"target_theme:{theme_id}"
    constraint_parts.append(Z[[theme_id]].rename(columns={theme_id: target_name}))
    C = pd.concat(constraint_parts, axis=1).astype(float)

    d = pd.Series(0.0, index=C.columns)
    d.loc[target_name] = 1.0

    Sigma_np = Sigma.to_numpy(dtype=float)
    C_np = C.to_numpy(dtype=float)
    d_np = d.to_numpy(dtype=float)

    Sigma_inv = np.linalg.pinv(Sigma_np, rcond=cfg.pinv_rcond)
    Sigma_inv_C = Sigma_inv @ C_np
    middle = C_np.T @ Sigma_inv_C
    multipliers = np.linalg.pinv(middle, rcond=cfg.pinv_rcond) @ d_np
    w_np = Sigma_inv_C @ multipliers

    weights = pd.Series(w_np, index=X.index, name=theme_id)
    constraint_values = pd.Series(C_np.T @ w_np, index=C.columns, name="value")
    constraint_errors = constraint_values - d

    barra_exposure = pd.Series(X.to_numpy(dtype=float).T @ w_np, index=X.columns, name=theme_id)
    theme_exposure = pd.Series(Z.to_numpy(dtype=float).T @ w_np, index=Z.columns, name=theme_id)

    variance = float(w_np @ Sigma_np @ w_np)
    gross = float(weights.abs().sum())
    long_exposure = float(weights.clip(lower=0.0).sum())
    short_exposure = float(-weights.clip(upper=0.0).sum())
    abs_weight = weights.abs()
    concentration_hhi = float(((abs_weight / gross) ** 2).sum()) if gross > 0 else np.nan
    other_abs = theme_exposure.drop(theme_id, errors="ignore").abs()

    diagnostics = pd.Series(
        {
            "theme_id": theme_id,
            "n_securities": len(weights),
            "n_barra_factors": X.shape[1],
            "n_theme_factors": Z.shape[1],
            "n_constraints": C.shape[1],
            "rank_C": _safe_matrix_rank(C_np),
            "rank_middle": _safe_matrix_rank(middle),
            "condition_C": _safe_condition_number(C_np),
            "condition_middle": _safe_condition_number(middle),
            "net_exposure": float(weights.sum()),
            "gross_exposure": gross,
            "long_exposure": long_exposure,
            "short_exposure": short_exposure,
            "portfolio_variance": variance,
            "portfolio_risk": float(np.sqrt(max(variance, 0.0))),
            "max_abs_barra_exposure": float(barra_exposure.abs().max()) if not barra_exposure.empty else 0.0,
            "target_theme_exposure": float(theme_exposure.loc[theme_id]),
            "target_exposure_error_abs": float(abs(theme_exposure.loc[theme_id] - 1.0)),
            "max_abs_other_theme_exposure": float(other_abs.max()) if not other_abs.empty else 0.0,
            "max_abs_constraint_error": float(constraint_errors.abs().max()),
            "concentration_hhi_abs_weight": concentration_hhi,
        }
    )

    constraint_table = pd.DataFrame({"target": d, "value": constraint_values, "error": constraint_errors})
    return {
        "weights": weights,
        "diagnostics": diagnostics,
        "constraints": constraint_table,
        "barra_exposure": barra_exposure,
        "theme_exposure": theme_exposure,
    }


def solve_theme_mimicking_portfolios(
    X_t: pd.DataFrame,
    Z_t: pd.DataFrame,
    Sigma_t: pd.DataFrame,
    cfg: FactorMimickingConfig,
) -> dict[str, Any]:
    Z = _ensure_str_index_columns(Z_t)
    weight_columns: dict[str, pd.Series] = {}
    diagnostics_rows: list[pd.Series] = []
    constraint_tables: dict[str, pd.DataFrame] = {}
    barra_exposures: dict[str, pd.Series] = {}
    theme_exposures: dict[str, pd.Series] = {}
    errors: dict[str, str] = {}

    for theme_id in Z.columns.astype(str):
        try:
            result = solve_one_mimicking_portfolio(X_t, Z, Sigma_t, theme_id, cfg)
            weight_columns[theme_id] = result["weights"]
            diagnostics_rows.append(result["diagnostics"])
            constraint_tables[theme_id] = result["constraints"]
            barra_exposures[theme_id] = result["barra_exposure"]
            theme_exposures[theme_id] = result["theme_exposure"]
        except Exception as exc:
            errors[theme_id] = f"{type(exc).__name__}: {exc}"
            if not cfg.continue_on_error:
                raise

    weights = pd.DataFrame(weight_columns).fillna(0.0).sort_index()
    diagnostics = pd.DataFrame(diagnostics_rows).set_index("theme_id") if diagnostics_rows else pd.DataFrame()
    barra_exposure = pd.DataFrame(barra_exposures).T if barra_exposures else pd.DataFrame()
    theme_exposure = pd.DataFrame(theme_exposures).T if theme_exposures else pd.DataFrame()
    error_series = pd.Series(errors, name="error", dtype="object")

    return {
        "mimicking_weights": weights,
        "mimicking_diagnostics": diagnostics,
        "constraint_tables": constraint_tables,
        "barra_exposure": barra_exposure,
        "theme_exposure": theme_exposure,
        "errors": error_series,
    }


## 項目10pklから履歴ウェイトを構築

`purified_snapshots.pkl` の各日付で `X_t`, `Z_t` を取り出し、`theme_residuals_e` から推定した対角リスクで複製ウェイトを作る。成功した日付のウェイトは必ず `MultiIndex(date, security_id)` にする。

In [ ]:
def _snapshot_to_exposures(snapshot: Mapping[str, Any], date: pd.Timestamp) -> tuple[pd.DataFrame, pd.DataFrame]:
    missing = [k for k in REQUIRED_SNAPSHOT_KEYS if k not in snapshot]
    if missing:
        raise KeyError(f"snapshot for {date.date()} is missing keys: {missing}")
    X_t = _ensure_str_index_columns(snapshot["X_t"]).astype(float)
    Z_t = _ensure_str_index_columns(snapshot["Z_t"]).astype(float)
    return X_t, Z_t


def _filter_history_dates(dates: Sequence[pd.Timestamp], cfg: FactorMimickingConfig) -> list[pd.Timestamp]:
    out = pd.DatetimeIndex(dates).normalize().sort_values().unique()
    if cfg.evaluation_start is not None:
        out = out[out >= _as_date(cfg.evaluation_start)]
    if cfg.evaluation_end is not None:
        out = out[out <= _as_date(cfg.evaluation_end)]
    return list(out)


def build_mimicking_weight_history_from_precomputed(
    bundle: Mapping[str, Any],
    cfg: FactorMimickingConfig,
    dates: Sequence[pd.Timestamp] | None = None,
) -> dict[str, Any]:
    snapshots = _normalise_datetime_key_dict(bundle["purified_snapshots"])
    theme_history = bundle["theme_return_history"]
    theme_residuals_e = _normalise_date_index(theme_history["theme_residuals_e"])
    theme_residuals_e.columns = theme_residuals_e.columns.astype(str)

    run_dates = _filter_history_dates(dates if dates is not None else snapshots.keys(), cfg)
    weight_frames: list[pd.DataFrame] = []
    diagnostic_frames: list[pd.DataFrame] = []
    risk_rows: list[pd.Series] = []
    error_rows: list[dict[str, Any]] = []

    for d in run_dates:
        try:
            if d not in snapshots:
                raise KeyError(f"purified_snapshots に {d.date()} がありません。forward-fillは行いません。")
            X_d, Z_d = _snapshot_to_exposures(snapshots[d], d)
            securities = X_d.index.intersection(Z_d.index)
            if securities.empty:
                raise ValueError("X_t と Z_t の共通銘柄が空です。")

            specific_variance, obs_count = estimate_specific_variance_from_theme_residuals(
                theme_residuals_e,
                d,
                securities,
                cfg,
            )
            Sigma_d = risk_matrix_from_specific_variance(specific_variance, cfg)
            result = solve_theme_mimicking_portfolios(X_d, Z_d, Sigma_d, cfg)
            W = result["mimicking_weights"]
            if W.empty:
                raise ValueError("No mimicking weights solved. Theme errors: " + result["errors"].to_dict().__repr__())

            W = W.copy()
            W.index = pd.MultiIndex.from_arrays(
                [np.repeat(pd.Timestamp(d), len(W)), W.index.astype(str)],
                names=["date", "security_id"],
            )
            weight_frames.append(W)

            diag = result["mimicking_diagnostics"].copy()
            if not diag.empty:
                diag.insert(0, "date", pd.Timestamp(d))
                diagnostic_frames.append(diag.reset_index().set_index(["date", "theme_id"]))

            risk_rows.append(
                pd.Series(
                    {
                        "date": pd.Timestamp(d),
                        "risk_window_end": pd.Timestamp(d),
                        "risk_window_observations": int(min(cfg.risk_lookback_days, len(theme_residuals_e.loc[theme_residuals_e.index <= d]))),
                        "n_specific_variance": int(len(specific_variance)),
                        "specific_variance_min": float(specific_variance.min()),
                        "specific_variance_median": float(specific_variance.median()),
                        "specific_variance_max": float(specific_variance.max()),
                        "obs_count_min": int(obs_count.min()),
                        "obs_count_median": float(obs_count.median()),
                    }
                )
            )

            if not result["errors"].empty:
                for theme_id, message in result["errors"].items():
                    error_rows.append({"date": pd.Timestamp(d), "theme_id": theme_id, "stage": "theme_solve", "message": message})
        except Exception as exc:
            error_rows.append({"date": pd.Timestamp(d), "theme_id": None, "stage": "date", "message": f"{type(exc).__name__}: {exc}"})
            if not cfg.continue_on_error:
                raise

    weights = pd.concat(weight_frames).sort_index() if weight_frames else _empty_weight_frame()
    diagnostics = pd.concat(diagnostic_frames).sort_index() if diagnostic_frames else pd.DataFrame()
    risk_diagnostics = pd.DataFrame(risk_rows).set_index("date").sort_index() if risk_rows else pd.DataFrame()
    errors = pd.DataFrame(error_rows).sort_values(["date", "stage", "theme_id"]).reset_index(drop=True) if error_rows else pd.DataFrame(columns=["date", "theme_id", "stage", "message"])

    return {
        "mimicking_weights": weights,
        "mimicking_diagnostics": diagnostics,
        "risk_diagnostics": risk_diagnostics,
        "history_errors": errors,
    }


def assert_mimicking_weight_history_ready(mimicking_weights: pd.DataFrame) -> None:
    if not isinstance(mimicking_weights.index, pd.MultiIndex):
        raise TypeError(
            "mimicking_weights must have MultiIndex index=(date, security_id). "
            f"Got index type={type(mimicking_weights.index).__name__}, names={mimicking_weights.index.names}."
        )
    if list(mimicking_weights.index.names) != ["date", "security_id"]:
        raise ValueError(
            "mimicking_weights index names must be ['date', 'security_id']. "
            f"Got {list(mimicking_weights.index.names)}."
        )
    if mimicking_weights.empty:
        raise ValueError("mimicking_weights is empty. Check history_errors before evaluating replication results.")


## 複製リターン評価

`weight_date < return_date` を満たす直近ウェイトを使い、Barra残差リターンおよび任意のトータルリターンで複製リターンを計算する。

In [ ]:
def _date_window(index: pd.DatetimeIndex, cfg: FactorMimickingConfig) -> pd.DatetimeIndex:
    dates = pd.DatetimeIndex(index).normalize().sort_values().unique()
    if cfg.evaluation_start is not None:
        dates = dates[dates >= _as_date(cfg.evaluation_start)]
    if cfg.evaluation_end is not None:
        dates = dates[dates <= _as_date(cfg.evaluation_end)]
    return dates


def _replication_metrics(replicated: pd.DataFrame, target: pd.DataFrame, annualization_days: int) -> pd.DataFrame:
    rows: list[dict[str, float | int | str]] = []
    for theme in replicated.columns.intersection(target.columns):
        aligned = pd.concat([replicated[theme], target[theme]], axis=1, keys=["replicated", "target"]).dropna()
        if aligned.empty:
            rows.append({"theme_id": theme, "observations": 0})
            continue
        diff = aligned["replicated"] - aligned["target"]
        target_var = float(aligned["target"].var(ddof=1)) if len(aligned) > 1 else np.nan
        beta = float(aligned.cov().loc["replicated", "target"] / target_var) if target_var and np.isfinite(target_var) and target_var > 0 else np.nan
        corr = float(aligned["replicated"].corr(aligned["target"])) if len(aligned) > 1 else np.nan
        rows.append(
            {
                "theme_id": theme,
                "observations": int(len(aligned)),
                "correlation": corr,
                "beta_to_target": beta,
                "r2_corr_squared": corr ** 2 if np.isfinite(corr) else np.nan,
                "mean_replicated_daily": float(aligned["replicated"].mean()),
                "mean_target_daily": float(aligned["target"].mean()),
                "mean_error_daily": float(diff.mean()),
                "tracking_error_daily": float(diff.std(ddof=1)) if len(diff) > 1 else np.nan,
                "tracking_error_annualized": float(diff.std(ddof=1) * np.sqrt(annualization_days)) if len(diff) > 1 else np.nan,
            }
        )
    return pd.DataFrame(rows).set_index("theme_id") if rows else pd.DataFrame()


def evaluate_replicated_returns(
    mimicking_weights: pd.DataFrame,
    barra_residuals_u: pd.DataFrame,
    theme_returns_g: pd.DataFrame,
    cfg: FactorMimickingConfig,
    stock_returns: pd.DataFrame | None = None,
) -> dict[str, pd.DataFrame]:
    assert_mimicking_weight_history_ready(mimicking_weights)

    residual = _normalise_date_index(barra_residuals_u)
    target = _normalise_date_index(theme_returns_g)
    residual.columns = residual.columns.astype(str)
    target.columns = target.columns.astype(str)

    stock = None
    if stock_returns is not None:
        stock = _normalise_date_index(stock_returns)
        stock.columns = stock.columns.astype(str)

    weight_dates = pd.DatetimeIndex(mimicking_weights.index.get_level_values("date").unique()).normalize().sort_values()
    return_index = residual.index.intersection(target.index)
    if stock is not None:
        return_index = return_index.intersection(stock.index)
    return_dates = _date_window(return_index, cfg)

    residual_rows: list[pd.Series] = []
    total_rows: list[pd.Series] = []
    coverage_rows: list[pd.Series] = []
    log_rows: list[dict[str, Any]] = []

    for d in return_dates:
        candidates = weight_dates[weight_dates < d]
        if len(candidates) == 0:
            log_rows.append({"date": d, "status": "skip", "reason": "no prior weight_date"})
            continue
        weight_date = candidates[-1]
        W = mimicking_weights.xs(weight_date, level="date")
        themes = W.columns.intersection(target.columns)
        securities_resid = W.index.intersection(residual.columns)
        if themes.empty or securities_resid.empty:
            log_rows.append({"date": d, "status": "skip", "reason": "empty common themes or residual securities", "weight_date": weight_date})
            continue

        W_resid = W.loc[securities_resid, themes].astype(float)
        u = residual.loc[d, securities_resid].astype(float)
        resid_valid = u.notna() & np.isfinite(u.to_numpy(dtype=float))
        if not resid_valid.any():
            log_rows.append({"date": d, "status": "skip", "reason": "no valid residual returns", "weight_date": weight_date})
            continue

        replicated_resid = W_resid.loc[resid_valid].mul(u.loc[resid_valid], axis=0).sum(axis=0)
        replicated_resid.name = d
        residual_rows.append(replicated_resid)

        gross = W_resid.abs().sum(axis=0).replace(0.0, np.nan)
        residual_coverage = W_resid.loc[resid_valid].abs().sum(axis=0) / gross
        coverage_piece = residual_coverage.rename("residual_abs_weight_coverage").to_frame().T
        coverage_piece.index = pd.MultiIndex.from_product([[d], coverage_piece.index], names=["date", "metric"])
        coverage_rows.append(coverage_piece)

        if stock is not None:
            securities_total = W.index.intersection(stock.columns)
            W_total = W.loc[securities_total, themes].astype(float)
            r = stock.loc[d, securities_total].astype(float)
            total_valid = r.notna() & np.isfinite(r.to_numpy(dtype=float))
            if total_valid.any():
                replicated_total = W_total.loc[total_valid].mul(r.loc[total_valid], axis=0).sum(axis=0)
                replicated_total.name = d
                total_rows.append(replicated_total)

                total_gross = W_total.abs().sum(axis=0).replace(0.0, np.nan)
                total_coverage = W_total.loc[total_valid].abs().sum(axis=0) / total_gross
                total_piece = total_coverage.rename("total_abs_weight_coverage").to_frame().T
                total_piece.index = pd.MultiIndex.from_product([[d], total_piece.index], names=["date", "metric"])
                coverage_rows.append(total_piece)

        log_rows.append({"date": d, "status": "ok", "weight_date": weight_date, "n_themes": len(themes), "n_residual_securities": int(resid_valid.sum())})

    replicated_residual = pd.DataFrame(residual_rows).sort_index() if residual_rows else pd.DataFrame()
    replicated_total = pd.DataFrame(total_rows).sort_index() if total_rows else pd.DataFrame()
    target_aligned = target.reindex(replicated_residual.index).reindex(columns=replicated_residual.columns)
    coverage = pd.concat(coverage_rows).sort_index() if coverage_rows else pd.DataFrame()
    evaluation_log = pd.DataFrame(log_rows)

    return {
        "replicated_returns_residual": replicated_residual,
        "replicated_returns_total": replicated_total,
        "target_theme_returns_g": target_aligned,
        "replication_metrics_residual": _replication_metrics(replicated_residual, target_aligned, cfg.annualization_days),
        "replication_metrics_total": _replication_metrics(replicated_total, target.reindex(replicated_total.index).reindex(columns=replicated_total.columns), cfg.annualization_days) if not replicated_total.empty else pd.DataFrame(),
        "daily_weight_coverage": coverage,
        "evaluation_log": evaluation_log,
    }


## ファクター寄与分解

任意で、直近ウェイトに対して `f_t` と `g_t` による Barra 寄与・純化テーマ寄与を確認する。Barra中立が効いていれば Barra 寄与はゼロ近傍になる。

In [ ]:
def decompose_factor_contributions_for_return_dates(
    mimicking_weights: pd.DataFrame,
    bundle: Mapping[str, Any],
    cfg: FactorMimickingConfig,
) -> dict[str, pd.DataFrame]:
    assert_mimicking_weight_history_ready(mimicking_weights)
    snapshots = _normalise_datetime_key_dict(bundle["purified_snapshots"])
    history = bundle["theme_return_history"]
    f_hist = _normalise_date_index(history["barra_factor_returns_f"])
    g_hist = _normalise_date_index(history["theme_returns_g"])

    weight_dates = pd.DatetimeIndex(mimicking_weights.index.get_level_values("date").unique()).normalize().sort_values()
    return_dates = _date_window(f_hist.index.intersection(g_hist.index), cfg)

    barra_rows: list[pd.Series] = []
    theme_rows: list[pd.Series] = []
    target_rows: list[pd.Series] = []
    log_rows: list[dict[str, Any]] = []

    for d in return_dates:
        candidates = weight_dates[weight_dates < d]
        if len(candidates) == 0:
            continue
        weight_date = candidates[-1]
        if weight_date not in snapshots:
            log_rows.append({"date": d, "weight_date": weight_date, "status": "skip", "reason": "missing snapshot for weight_date"})
            continue

        X_w, Z_w = _snapshot_to_exposures(snapshots[weight_date], weight_date)
        W = mimicking_weights.xs(weight_date, level="date")
        securities = W.index.intersection(X_w.index).intersection(Z_w.index)
        factors = X_w.columns.intersection(f_hist.columns)
        themes = W.columns.intersection(Z_w.columns).intersection(g_hist.columns)
        if securities.empty or factors.empty or themes.empty:
            log_rows.append({"date": d, "weight_date": weight_date, "status": "skip", "reason": "empty common exposure set"})
            continue

        W = W.loc[securities, themes].astype(float)
        X = X_w.loc[securities, factors].astype(float)
        Z = Z_w.loc[securities, themes].astype(float)
        f = f_hist.loc[d, factors].astype(float)
        g = g_hist.loc[d, themes].astype(float)

        barra_exp = X.T @ W
        theme_exp = Z.T @ W
        barra_contribution = barra_exp.T @ f
        theme_contribution = theme_exp.T @ g
        target_contribution = pd.Series({theme: float(theme_exp.loc[theme, theme] * g.loc[theme]) for theme in themes})
        barra_contribution.name = d
        theme_contribution.name = d
        target_contribution.name = d
        barra_rows.append(barra_contribution)
        theme_rows.append(theme_contribution)
        target_rows.append(target_contribution)
        log_rows.append({"date": d, "weight_date": weight_date, "status": "ok", "n_themes": len(themes), "n_securities": len(securities)})

    return {
        "barra_factor_contribution": pd.DataFrame(barra_rows).sort_index() if barra_rows else pd.DataFrame(),
        "pure_theme_factor_contribution": pd.DataFrame(theme_rows).sort_index() if theme_rows else pd.DataFrame(),
        "target_theme_contribution": pd.DataFrame(target_rows).sort_index() if target_rows else pd.DataFrame(),
        "factor_contribution_log": pd.DataFrame(log_rows),
    }


## 履歴型の主実行セル

項目10の事前計算pklをロードし、履歴ウェイト、複製リターン、診断を作成する。`stock_returns` が未指定の場合、トータルリターン評価だけをスキップする。

In [ ]:
if RUN_PRECOMPUTED_HISTORY:
    precomputed_bundle = load_item10_precomputed_bundle(cfg.precompute_dir)
    print("loaded precompute_dir:", _resolve_path(cfg.precompute_dir))
    display(pd.Series(precomputed_bundle["metadata"]).to_frame("value"))

    history_result = build_mimicking_weight_history_from_precomputed(precomputed_bundle, cfg)
    mimicking_weights = history_result["mimicking_weights"]
    mimicking_diagnostics = history_result["mimicking_diagnostics"]
    risk_diagnostics = history_result["risk_diagnostics"]
    history_errors = history_result["history_errors"]

    print("mimicking_weights shape:", mimicking_weights.shape)
    print("mimicking_weights index type:", type(mimicking_weights.index).__name__)
    print("mimicking_weights index names:", mimicking_weights.index.names)
    if not history_errors.empty:
        print("history_errors:")
        display(history_errors.head(50))

    assert_mimicking_weight_history_ready(mimicking_weights)

    stock_returns = _load_input("stock_returns", required=False)
    theme_history = precomputed_bundle["theme_return_history"]
    replication_result = evaluate_replicated_returns(
        mimicking_weights=mimicking_weights,
        barra_residuals_u=theme_history["barra_residuals_u"],
        theme_returns_g=theme_history["theme_returns_g"],
        cfg=cfg,
        stock_returns=stock_returns,
    )

    replicated_returns_residual = replication_result["replicated_returns_residual"]
    replicated_returns_total = replication_result["replicated_returns_total"]
    target_theme_returns_g = replication_result["target_theme_returns_g"]
    replication_metrics_residual = replication_result["replication_metrics_residual"]
    replication_metrics_total = replication_result["replication_metrics_total"]

    if RUN_FACTOR_CONTRIBUTION_CHECK:
        factor_contribution_result = decompose_factor_contributions_for_return_dates(mimicking_weights, precomputed_bundle, cfg)
    else:
        factor_contribution_result = None
else:
    print("RUN_PRECOMPUTED_HISTORY=False のため、履歴型主実行セルはスキップしました。")


## 出力確認

In [ ]:
if RUN_PRECOMPUTED_HISTORY:
    display(mimicking_diagnostics.head())
    display(risk_diagnostics.head())
    display(replication_metrics_residual)
    if not replication_metrics_total.empty:
        display(replication_metrics_total)
    display(replication_result["evaluation_log"].tail())
    if factor_contribution_result is not None:
        display(factor_contribution_result["barra_factor_contribution"].tail())
        display(factor_contribution_result["pure_theme_factor_contribution"].tail())


## 補助: 単日スナップショット確認

既定では使わない。特定日だけ制約状態を見たい場合に `RUN_ASOF_SNAPSHOT=True` と `cfg.asof_date` を設定して実行する。履歴評価にはこの出力を渡さない。

In [ ]:
if RUN_ASOF_SNAPSHOT:
    precomputed_bundle_asof = load_item10_precomputed_bundle(cfg.precompute_dir)
    snapshots_asof = precomputed_bundle_asof["purified_snapshots"]
    d = _as_date(cfg.asof_date)
    if d not in snapshots_asof:
        raise KeyError(f"purified_snapshots に {d.date()} がありません。")
    X_t, Z_t = _snapshot_to_exposures(snapshots_asof[d], d)
    securities = X_t.index.intersection(Z_t.index)
    specific_variance_t, obs_count_t = estimate_specific_variance_from_theme_residuals(
        precomputed_bundle_asof["theme_return_history"]["theme_residuals_e"],
        d,
        securities,
        cfg,
    )
    Sigma_t = risk_matrix_from_specific_variance(specific_variance_t, cfg)
    asof_result = solve_theme_mimicking_portfolios(X_t, Z_t, Sigma_t, cfg)
    mimicking_weights_t = asof_result["mimicking_weights"]
    mimicking_diagnostics_t = asof_result["mimicking_diagnostics"]
    display(mimicking_diagnostics_t)
    display(mimicking_weights_t.head())


## 受入チェック

履歴型 `mimicking_weights` に対して、MultiIndex形式と制約誤差を確認する。全日失敗時は `history_errors` を確認してから停止する。

In [ ]:
if RUN_PRECOMPUTED_HISTORY:
    assert_mimicking_weight_history_ready(mimicking_weights)
    assert list(mimicking_weights.index.names) == ["date", "security_id"]
    assert not mimicking_diagnostics.empty, "mimicking_diagnostics is empty."

    assert mimicking_diagnostics["target_exposure_error_abs"].max() < cfg.constraint_tol, "z_k' w_k must be close to 1."
    assert mimicking_diagnostics["net_exposure"].abs().max() < cfg.constraint_tol, "sum(w_k) must be close to 0."
    assert mimicking_diagnostics["max_abs_barra_exposure"].max() < cfg.constraint_tol, "X_t' w_k must be close to 0."
    if cfg.neutralize_other_themes:
        assert mimicking_diagnostics["max_abs_other_theme_exposure"].max() < cfg.constraint_tol, "Z_{-k}' w_k must be close to 0."

    assert not replicated_returns_residual.empty, "replicated_returns_residual is empty."
    assert replicated_returns_residual.index.min() > mimicking_weights.index.get_level_values("date").min(), "returns must be evaluated after the first weight date."
    print("Acceptance checks passed for precomputed history mode.")


## 次の確認ポイント

- `mimicking_weights.index.names` が `['date', 'security_id']` になっていることを確認する。
- `history_errors` が出る場合は、該当日の `theme_residuals_e` 不足、snapshot欠損、rank不足、または制約過多を確認する。
- `replication_metrics_residual` の correlation、beta、tracking error で `g_t` の複製精度を確認する。
- `barra_factor_contribution` がゼロ近傍なら、Barra既存因子中立が効いている。
- `stock_returns` を指定した場合のみ、`replicated_returns_total` と `replication_metrics_total` も確認する。